# Setup Notebook

In [42]:
#path configuration
import os
import sys

#data manipulation
import pandas as pd
import numpy as np

#sklearn libraries
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import MinMaxScaler

#statsmodels
import statsmodels.api as sm
from statsmodels.formula.api import glm
from statsmodels.genmod.families import Binomial, Gamma, InverseGaussian, Gaussian
from statsmodels.genmod.families.links import logit, log

#model evaluation
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report
from sklearn.calibration import calibration_curve

#other statistics
import scipy.stats as stats
import itertools

#data visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#import warning
import warnings
warnings.filterwarnings("ignore")

#reference seed
seed = 42
np.random.seed(seed)

# Import Notebooks and Dataframe

In [43]:
# Import DataFrame
df_exp = pd.read_csv("data/df_exp_simulation.csv").drop(columns = "Unnamed: 0")
print(f"Amount of experiments: {len(df_exp)}")

#Create period column 
# df_exp["Period"] = df_exp.apply(lambda row: f"({row['initial_date']}-{row['final_date']}]", axis = 1)

#adjust the model name
df_exp.replace({"dt": "DT",
                "rf": "RF",
                "lgbm": "LGBM",
                "xgb": "XGB"},
                inplace = True)

#consistency check
print("Duplicados: ", df_exp.duplicated().sum())

#Visualize dataframe
df_exp.head()

Amount of experiments: 1728
Duplicados:  0


,initial_date,final_date,time_interval,create_lag_features_abs,create_lag_features_rel,list_lag_features,list_columns_filter,seed,test_size,remove_correlation,...,best_params,scoring,max_evals,timeout,cv,score_train,score_test,score_train_rank,score_test_rank,Period
0,0,100,1,True,False,[3],NaN,42,0.25,False,...,"{'max_depth': 6, 'min_samples_leaf': 0.005, 'm...",roc_auc,60,600,5,0.768492,0.790233,173.0,1.0,(0-100]
1,0,100,1,True,False,[3],NaN,42,0.25,False,...,"{'alpha': 0.0, 'colsample_bytree': 0.789966965...",roc_auc,60,3600,5,0.768348,0.790086,175.0,2.0,(0-100]
2,0,100,1,True,False,[3],NaN,42,0.25,True,...,"{'max_depth': 6, 'min_samples_leaf': 0.005, 'm...",roc_auc,60,600,5,0.768943,0.789737,170.0,3.0,(0-100]
3,0,100,1,True,False,[5],NaN,42,0.25,True,...,"{'alpha': 0.0, 'colsample_bytree': 0.620714883...",roc_auc,60,3600,5,0.765992,0.789701,191.0,4.0,(0-100]
4,0,100,5,True,False,[3],NaN,42,0.25,True,...,"{'max_depth': 5, 'min_samples_leaf': 0.005, 'm...",roc_auc,60,600,5,0.767744,0.789695,178.0,5.0,(0-100]


# Boxplot of Models Performance

In [38]:
fig = px.box(df_exp.replace({"(0-100]": "First", 
                                 "(100-200]": "Second",
                                 "(200-300]": "Third"}),
       title = "Boxplot of Models Performance",
       x = "model",
       y = "score_test",
       facet_col = "Period",
       color = "model",
       category_orders = {"model": ["DT", "RF", "XGB", "LGBM"]},
       height = 500,
       width = 1200,
       color_discrete_map={
                     "DT": "black",
                     "RF": "dimgray",     # Darker gray
                     "XGB": "gray",       # Medium gray
                     "LGBM": "gainsboro"  # Light gray
                 },)

# Increase marker size by 5%
fig.update_traces(marker=dict(size=8))  # Adjust the size as needed (default is 10)

# Hide legend
fig.update_layout(showlegend=False)
fig.update_annotations(font=dict(size=18))

# Ensure all x-axis titles, ticks, and grids are the same for all subplots
for i in range(1, len(fig.layout.annotations) + 1):
    fig.update_xaxes(title_text="Model", title_font=dict(size=20), tickfont=dict(size=16),
                     showgrid=True, gridcolor="lightgray", row=1, col=i)
    fig.update_yaxes(tickfont=dict(size=16),
                     showgrid=True, gridcolor="lightgray", row=1, col=i)

# Update the layout as before
fig.update_layout(
    title={'font': {'size': 24}, 'x': 0.5},
    yaxis_title={'text': 'Testing Score', 'font': {'size': 20}},
    legend=dict(
        title='Model',
        font=dict(size=16),
        title_font=dict(size=18)
    ),
    plot_bgcolor='white',
    paper_bgcolor='white'
)

# Add borders around each subplot
for i, annotation in enumerate(fig.layout.annotations):
    x_start, x_end = fig.layout[f"xaxis{i+1}"].domain
    y_start, y_end = fig.layout[f"yaxis{i+1}"].domain
    
    # Add a border to each subplot
    fig.add_shape(type="rect",
                  xref="paper", yref="paper",
                  x0=x_start, y0=y_start, x1=x_end, y1=y_end,
                  line=dict(color="lightgray", width=1.25),  # Border color and width
                  fillcolor="rgba(0,0,0,0)")  # No fill, transparent



fig.show()

In [41]:
df_exp.groupby("Period").score_test.mean()

Period
(0-100]      0.750466
(100-200]    0.781468
(200-300]    0.965991
Name: score_test, dtype: float64

# Distribution of Training and Testing Score

## Scatterplot of the Training and Testing Score

In [3]:
#create scatterplot figiure
fig = px.scatter(df_exp,
                 title = "Scatterplot of Training and Testing Scores",
                 x = "score_train",
                 y = "score_test",
                 color = "model",
                 symbol = "model",
                 category_orders = {"model": ["DT", "RF", "XGB", "LGBM"]},
                 facet_col = "Period",
                 opacity = 0.75,
                 color_discrete_sequence=px.colors.qualitative.G10,
                 width = 1200,
                 height = 500)

subplot_titles = fig.layout.annotations
for i, annotation in enumerate(subplot_titles):
    # Add the diagonal hash line to each subplot
    fig.add_shape(
        type='line',
        x0=0.7, y0=0.7,
        x1=1.0, y1=1.0,
        xref=f'x{i+1}', yref=f'y{i+1}',
        line=dict(color='black', width=2, dash='dash')
    ),
    annotation['font'] = dict(size=18)  # You can adjust the size as needed
    
# Apply the xaxis and yaxis title and font to all subplots
for axis in fig.layout:
    if axis.startswith('xaxis'):
        fig.layout[axis].title = {'text': 'Training Score', 'font': {'size': 20}}
        fig.layout[axis].tickfont = {'size': 16}  # Set tick font size if needed
        fig.layout[axis].tickvals = [0.7, 0.8, 0.9, 1.0]  # Set specific ticks for x-axis
    elif axis.startswith('yaxis'):
        fig.layout[axis].tickvals = [0.7, 0.8, 0.9, 1.0]  # Set specific ticks for x-axis
    #     fig.layout[axis].title = {'text': 'Test Score (mean)', 'font': {'size': 20}}
    #     fig.layout[axis].tickfont = {'size': 16}  # Set tick font size if needed

#update layout
fig.update_layout(
    title={'font': {'size': 24}, 'x': 0.5},
    xaxis_title={'text': 'Training Score', 'font': {'size': 20}},
    yaxis_title={'text': 'Testing Score', 'font': {'size': 20}},
    xaxis={'tickfont': {'size': 16}},
    yaxis={'tickfont': {'size': 16}},
    legend=dict(
        title='Model',
        font=dict(size=14),  # Increase the legend text size
        title_font=dict(size=16)),  # Increase the legend title size,  # Increase legend text size
    # plot_bgcolor='white',  # Background color of the plot area
    # paper_bgcolor='white'  # Background color of the entire figure
)


In [462]:
import plotly.express as px

# Scatterplot setup with custom colors and larger markers
fig = px.scatter(df_exp.replace({"(0-100]": "First", 
                                 "(100-200]": "Second",
                                 "(200-300]": "Third"}),
                 title="Scatterplot of Training and Testing Scores",
                 x="score_train",
                 y="score_test",
                 color="model",
                 symbol="model",
                 category_orders={"model": ["DT", "RF", "XGB", "LGBM"]},
                 facet_col="Period",
                 opacity=0.75,
                 color_discrete_map={
                     "DT": "black",
                     "RF": "dimgray",     # Darker gray
                     "XGB": "gray",       # Medium gray
                     "LGBM": "gainsboro"  # Light gray
                 },

                 width=1200,
                 height=500)

# Increase marker size by 5%
fig.update_traces(marker=dict(size=8))  # Adjust the size as needed (default is 10)

# Add diagonal lines and update annotation fonts
subplot_titles = fig.layout.annotations
for i, annotation in enumerate(subplot_titles):
    # Add the diagonal hash line to each subplot
    fig.add_shape(
        type='line',
        x0=0.7, y0=0.7,
        x1=1.0, y1=1.0,
        xref=f'x{i+1}', yref=f'y{i+1}',
        line=dict(color='black', width=2, dash='dash')
    ),
    annotation['font'] = dict(size=18)

# Add borders around each subplot
for i, annotation in enumerate(fig.layout.annotations):
    x_start, x_end = fig.layout[f"xaxis{i+1}"].domain
    y_start, y_end = fig.layout[f"yaxis{i+1}"].domain
    
    # Add a border to each subplot
    fig.add_shape(type="rect",
                  xref="paper", yref="paper",
                  x0=x_start, y0=y_start, x1=x_end, y1=y_end,
                  line=dict(color="lightgray", width=1.25),  # Border color and width
                  fillcolor="rgba(0,0,0,0)")  # No fill, transparent

# Ensure all x-axis titles, ticks, and grids are the same for all subplots
for i in range(1, len(fig.layout.annotations) + 1):
    fig.update_xaxes(title_text="Training Score", title_font=dict(size=20), tickfont=dict(size=16),
                     showgrid=True, gridcolor="lightgray", row=1, col=i)
    fig.update_yaxes(tickfont=dict(size=16),
                     showgrid=True, gridcolor="lightgray", row=1, col=i)

# Update the layout as before
fig.update_layout(
    title={'font': {'size': 24}, 'x': 0.5},
    yaxis_title={'text': 'Testing Score', 'font': {'size': 20}},
    legend=dict(
        title='Model',
        font=dict(size=16),
        title_font=dict(size=18)
    ),
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.show()


In [427]:
# df_exp_grouped = df_exp.groupby(['model', 'Period']).agg(
#     mean_score_train=('score_train', 'mean'),
#     mean_score_test=('score_test', 'mean'),
#     std_score_train=('score_train', 'std'),
#     std_score_test=('score_test', 'std')
# ).reset_index()

# fig = px.scatter(
#     df_exp_grouped, 
#     x='mean_score_train', 
#     y='mean_score_test', 
#     error_x='std_score_train', 
#     error_y='std_score_test',
#     color='model',
#     facet_col='Period',
#     title = "Scatterplot of mean model scores and standard deviation",
#     opacity = 1,
#     category_orders = {"model": ["dt", "rf", "xgb", "lgbm"]},
#     color_discrete_sequence=px.colors.qualitative.G10
# )

# subplot_titles = fig.layout.annotations
# for i, annotation in enumerate(subplot_titles):
#     # Add the diagonal hash line to each subplot
#     fig.add_shape(
#         type='line',
#         x0=0.7, y0=0.7,
#         x1=1.0, y1=1.0,
#         xref=f'x{i+1}', yref=f'y{i+1}',
#         line=dict(color='black', width=2, dash='dash')
#     ),
#     annotation['font'] = dict(size=18)  # You can adjust the size as needed
    
# # Apply the xaxis and yaxis title and font to all subplots
# for axis in fig.layout:
#     if axis.startswith('xaxis'):
#         fig.layout[axis].title = {'text': 'Train Score (mean)', 'font': {'size': 20}}
#         fig.layout[axis].tickfont = {'size': 16}  # Set tick font size if needed
#     elif axis.startswith('yaxis'):
#         fig.layout[axis].title = {'text': 'Test Score (mean)', 'font': {'size': 20}}
#         fig.layout[axis].tickfont = {'size': 16}  # Set tick font size if needed

# #update layout
# fig.update_layout(
#     title={'font': {'size': 24}},
#     xaxis_title={'text': 'Train Score (mean)', 'font': {'size': 20}},
#     yaxis_title={'text': 'Test Score (mean)', 'font': {'size': 20}},
#     xaxis={'tickfont': {'size': 16}},
#     yaxis={'tickfont': {'size': 16}},
#     legend=dict(
#         title='Model',
#         font=dict(size=14),  # Increase the legend text size
#         title_font=dict(size=16)),  # Increase the legend title size,  # Increase legend text size
# #     plot_bgcolor='white',  # Background color of the plot area
# #     paper_bgcolor='white'  # Background color of the entire figure
# )

# fig.show()

In [370]:
df_exp_grouped[['model', 'Period', 
                'mean_score_train', "std_score_train",
                'mean_score_test', 'std_score_test']].round(3).to_latex(
                )

'\\begin{tabular}{lllrrrr}\n\\toprule\n{} & model &     Period &  mean\\_score\\_train &  std\\_score\\_train &  mean\\_score\\_test &  std\\_score\\_test \\\\\n\\midrule\n0  &    DT &    (0-100] &             0.742 &            0.026 &            0.742 &           0.017 \\\\\n1  &    DT &  (100-200] &             0.776 &            0.031 &            0.775 &           0.020 \\\\\n2  &    DT &  (200-300] &             0.961 &            0.029 &            0.957 &           0.031 \\\\\n3  &  LGBM &    (0-100] &             0.802 &            0.080 &            0.747 &           0.034 \\\\\n4  &  LGBM &  (100-200] &             0.832 &            0.073 &            0.778 &           0.035 \\\\\n5  &  LGBM &  (200-300] &             0.978 &            0.025 &            0.972 &           0.028 \\\\\n6  &    RF &    (0-100] &             0.761 &            0.024 &            0.769 &           0.014 \\\\\n7  &    RF &  (100-200] &             0.793 &            0.024 &            0.797 &   

## Distribution of Model Performance

In [371]:
px.box(df_exp,
       y = "score_test",
       color = "model",
       facet_col = "Period",
       category_orders = {"model": ["dt", "rf", "xgb", "catb", "lgbm"]},
       title = "Models Performance Distribution",
       width = 1200)

## Distribution of Models Performance, According to Parameters

In [372]:
# for parameter in ["time_interval",
#                   "create_lag_features_abs", "list_lag_features",
#                   "remove_correlation", "apply_fa", "apply_rebalance_strategy"]:
#   px.box(df_exp.fillna("None"),
#         y = "score_test",
#         color = "model",
#         facet_row = "Period",
#         facet_col = parameter,
#         category_orders = {"model": ["dt", "rf", "xgb", "catb", "lgbm"],
#                            "time_interval": [1, 5, 10, 20],
#                            "list_lag_features": ["None", "[3]", "[5]"],
#                            "apply_rebalance_strategy": ["None", "under_sampling", "over_sampling"]},
#         title = f"Models Performance Distribution, by {parameter}",
#         height = 1000, width = df_exp[parameter].nunique()*400).show()

In [373]:
#create empty list for dataframes
df_exp_grouped_parameter_complete = []

#iterate over parameters
for parameter in  ["time_interval",
                   "create_lag_features_abs", "list_lag_features",
                   "remove_correlation", "apply_fa", "apply_rebalance_strategy"]:
    
    #group the dataframe
    df_exp_grouped_parameter = df_exp.fillna("None").groupby([parameter, 'model'], as_index = False).agg(
        mean_score_test=('score_test', 'mean'),
        std_score_test=('score_test', 'std'))
    
    #rename columns
    df_exp_grouped_parameter.rename(columns = {"mean_score_test": "Mean",
                                               "std_score_test": "Standard Deviation",
                                               "model": "Model",
                                               parameter: "Category",
                                               "model": "Model"},
                                               inplace = True)

    #create column of the parameter
    df_exp_grouped_parameter["Preprocessing Technique"] = parameter
    
    #add the dataframe over the list
    df_exp_grouped_parameter_complete.append(df_exp_grouped_parameter)

#transform the list into dataframe
df_exp_grouped_parameter_complete = pd.concat(df_exp_grouped_parameter_complete)

#reorder columns
df_exp_grouped_parameter_complete = df_exp_grouped_parameter_complete[["Preprocessing Technique", "Category", "Model", "Mean", "Standard Deviation"]].round(3)

#print dataframe
df_exp_grouped_parameter_complete

,Preprocessing Technique,Category,Model,Mean,Standard Deviation
0,time_interval,1,DT,0.820,0.096
1,time_interval,1,LGBM,0.839,0.103
2,time_interval,1,RF,0.844,0.085
3,time_interval,1,XGB,0.838,0.105
4,time_interval,5,DT,0.826,0.098
...,...,...,...,...,...
7,apply_rebalance_strategy,over_sampling,XGB,0.796,0.128
8,apply_rebalance_strategy,under_sampling,DT,0.822,0.098
9,apply_rebalance_strategy,under_sampling,LGBM,0.844,0.094
10,apply_rebalance_strategy,under_sampling,RF,0.842,0.089


In [374]:
df_exp_grouped_parameter_complete.to_latex()

'\\begin{tabular}{llllrr}\n\\toprule\n{} &   Preprocessing Technique &        Category & Model &   Mean &  Standard Deviation \\\\\n\\midrule\n0  &             time\\_interval &               1 &    DT &  0.820 &               0.096 \\\\\n1  &             time\\_interval &               1 &  LGBM &  0.839 &               0.103 \\\\\n2  &             time\\_interval &               1 &    RF &  0.844 &               0.085 \\\\\n3  &             time\\_interval &               1 &   XGB &  0.838 &               0.105 \\\\\n4  &             time\\_interval &               5 &    DT &  0.826 &               0.098 \\\\\n5  &             time\\_interval &               5 &  LGBM &  0.835 &               0.105 \\\\\n6  &             time\\_interval &               5 &    RF &  0.845 &               0.088 \\\\\n7  &             time\\_interval &               5 &   XGB &  0.834 &               0.106 \\\\\n8  &             time\\_interval &              10 &    DT &  0.825 &               0.097

# Interaction Visualization

In [375]:
def create_interaction_graph(df_exp,
                             x,
                             color):
        
        #filter the columns
        df_int = df_exp[["Period", "time_interval", "list_lag_features", "remove_correlation", 
                        "apply_fa", "apply_rebalance_strategy", "model", "score_test"]].copy().fillna("None")

        #group the information
        df_int = df_int.groupby(["Period", x, color], as_index = False)[["score_test"]].mean()

        #plot the graph
        fig = px.line(df_int,
                        x = x,
                        y = "score_test",
                        color = color,
                        facet_col = "Period",
                        title = f"Interaction between {x} and {color}",
                        height = 400, width = 1600)
        
        fig.show()

In [423]:
# columns_interaction = ["time_interval", "list_lag_features", "remove_correlation", "apply_fa", "apply_rebalance_strategy", "model"]

# # Generate all pairs of values
# column_pairs = list(itertools.combinations(columns_interaction, 2))

# #iterate all values to plot iterations
# for (column1, column2) in column_pairs:
#     create_interaction_graph(df_exp,
#                         x = column1,
#                         color = column2)

# Correlation

In [377]:
def sort_dataframe(df_exp, period):
    if period == 1:
        period_str = "(0-100]"
    elif period == 2:
        period_str = "(100-200]"
    else:
        period_str = "(200-300]"

    df_period = df_exp.loc[df_exp.Period == period_str].copy()

    df_period.sort_values(['time_interval',
       'create_lag_features_abs', 'create_lag_features_rel', 'list_lag_features',
       'remove_correlation', 'remove_correlation_params', 'apply_fa',
       'fa_params', 'apply_rebalance_strategy', 'rebalance_params', 'model'], inplace = True)
    
    df_period.reset_index(inplace = True, drop = True)

    return df_period

df_period1 = sort_dataframe(df_exp, 1)
df_period2 = sort_dataframe(df_exp, 2)
df_period3 = sort_dataframe(df_exp, 3)

In [378]:
pd.concat([df_period1[["score_test"]], df_period2[["score_test"]], df_period3[["score_test"]]], axis = 1).corr()

,score_test,score_test,score_test
score_test,1.000000,0.846805,0.125242
score_test,0.846805,1.000000,0.210045
score_test,0.125242,0.210045,1.000000


# Regression Modeling

## Data Prep

In [381]:
#slice dataframe
df_reg = df_exp[["Period", "time_interval", "list_lag_features", "remove_correlation", 
                 "apply_fa", "apply_rebalance_strategy", "model", "score_test"]].copy()

#convert the type of the time_interval dataframe
# df_reg.time_interval = df_reg.time_interval.astype(str)

#drop outliers of first period
df_reg = df_reg.drop(df_reg.loc[df_reg["Period"] == "(0-100]"].sort_values("score_test").head(3).index)

#get dummy variables
df_reg = pd.get_dummies(df_reg, columns = ['time_interval', 'list_lag_features', 'remove_correlation',
                                            'apply_fa', 'apply_rebalance_strategy', 'model'], dummy_na = True)

#drop constant columns
df_reg.drop(columns = ["time_interval_nan", "apply_fa_nan", "remove_correlation_nan", "apply_fa_nan", "model_nan"], inplace = True)

#drop reference columns
df_reg.drop(columns = ["time_interval_1.0", "list_lag_features_nan", "remove_correlation_False", "apply_fa_False", 
                       "apply_rebalance_strategy_nan", "model_DT"], inplace = True)

df_reg.head(2)

,Period,score_test,time_interval_5.0,time_interval_10.0,time_interval_20.0,list_lag_features_[3],list_lag_features_[5],remove_correlation_True,apply_fa_True,apply_rebalance_strategy_over_sampling,apply_rebalance_strategy_under_sampling,model_LGBM,model_RF,model_XGB
0,(0-100],0.790233,0,0,0,1,0,0,0,0,0,0,1,0
1,(0-100],0.790086,0,0,0,1,0,0,0,0,0,0,0,1


In [249]:
# df_reg.to_csv("df_regression.csv")

## Linear Regression - sklearn

In [382]:
def create_linear_regression(df,
                            period):
    
    #slice dataframe
    if period == "first":
        df_copy = df.loc[df.Period == "(0-100]"].copy()

    elif period == "second":
        df_copy = df.loc[df.Period == "(100-200]"].copy()

    elif period == "third":
        df_copy = df.loc[df.Period == "(200-300]"].copy()

    #drop the period column
    df_copy.drop(columns = ["Period"], inplace = True)

    #Separate X and y
    X = df_copy.drop(columns = ["score_test"]).copy()
    y = df_copy[["score_test"]].copy()

    #create the linear regression model
    lin = LinearRegression().fit(X, y)

    #create predictions
    y_pred = lin.predict(X)

    return lin, X, y, y_pred

In [383]:
#obtain modeling information
lin1, X1, y1, y_pred1 = create_linear_regression(df_reg, period = "first")
lin2, X2, y2, y_pred2 = create_linear_regression(df_reg, period = "second")
lin3, X3, y3, y_pred3 = create_linear_regression(df_reg, period = "third")

#create empty dataframe
df_lin_results = pd.DataFrame()

#iterate for each period
for (period, lin, X, y, y_pred) in zip (["first", "second", "third"], [lin1, lin2, lin3], [X1, X2, X3], 
                                        [y1, y2, y3], [y_pred1, y_pred2, y_pred3]):
    
    #create empty dataframe with coeficientes
    df_lin_coef = pd.DataFrame(columns= X.columns,
                               data = lin.coef_)

    #calculate RMSE
    rmse = root_mean_squared_error(y, y_pred)

    #create dataframe with overall results
    df_results = pd.DataFrame(data = {"Period": period,
                                      "RMSE": rmse},
                                      index = [0])
    
    #concatenate iteration results
    df_results = pd.concat([df_results, df_lin_coef], axis = 1)
    
    #concatenate results
    df_lin_results = pd.concat([df_lin_results, df_results], axis = 0)
    
#final adjustment
df_lin_results.reset_index(drop = True, inplace = True)
df_lin_results = df_lin_results.round(3)

#visualization
df_lin_results

,Period,RMSE,time_interval_5.0,time_interval_10.0,time_interval_20.0,list_lag_features_[3],list_lag_features_[5],remove_correlation_True,apply_fa_True,apply_rebalance_strategy_over_sampling,apply_rebalance_strategy_under_sampling,model_LGBM,model_RF,model_XGB
0,first,0.019,-0.005,-0.009,-0.016,0.005,0.002,-0.002,-0.016,-0.039,-0.008,0.005,0.027,0.004
1,second,0.021,0.002,0.000,0.001,0.008,0.006,0.005,-0.023,-0.038,-0.004,0.003,0.022,0.001
2,third,0.019,0.003,-0.002,-0.003,0.020,0.012,-0.018,-0.033,-0.001,-0.002,0.015,0.006,0.015


## Linear Regression - StatsModels

In [384]:
def run_linear_statsmodels(X, y, columns_remove = None):

    X_copy = X.copy(deep = True)

    if columns_remove != None:
        X_copy = X_copy.drop(columns = columns_remove)

    # Add a constant to the model (for the intercept)
    X_copy = sm.add_constant(X_copy)

    # Fit the linear regression model
    model = sm.OLS(y, X_copy).fit()

    # Get a summary of the results
    print(model.summary())

    return model, X_copy, y

In [385]:
lin1, X1_after, y1_after = run_linear_statsmodels(X1, y1)

                            OLS Regression Results                            
Dep. Variable:             score_test   R-squared:                       0.563
Model:                            OLS   Adj. R-squared:                  0.554
Method:                 Least Squares   F-statistic:                     60.21
Date:                Tue, 08 Oct 2024   Prob (F-statistic):           1.48e-92
Time:                        14:51:58   Log-Likelihood:                 1447.7
No. Observations:                 573   AIC:                            -2869.
Df Residuals:                     560   BIC:                            -2813.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

In [386]:
lin2, X2_after, y2_after = run_linear_statsmodels(X2, y2)
# lin2, X2_after, y2_after = run_linear_statsmodels(X2, y2, columns_remove = ["time_interval_5.0", "time_interval_10.0", "time_interval_20.0"])

                            OLS Regression Results                            
Dep. Variable:             score_test   R-squared:                       0.539
Model:                            OLS   Adj. R-squared:                  0.529
Method:                 Least Squares   F-statistic:                     54.76
Date:                Tue, 08 Oct 2024   Prob (F-statistic):           1.97e-86
Time:                        14:51:58   Log-Likelihood:                 1407.9
No. Observations:                 576   AIC:                            -2790.
Df Residuals:                     563   BIC:                            -2733.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

In [387]:
lin3, X3_after, y3_after = run_linear_statsmodels(X3, y3)
# lin3, X3_after, y3_after = run_linear_statsmodels(X3, y3, columns_remove = ["time_interval_5.0", "time_interval_10.0", "time_interval_20.0", "apply_rebalance_strategy_over_sampling", "apply_rebalance_strategy_under_sampling"])

                            OLS Regression Results                            
Dep. Variable:             score_test   R-squared:                       0.559
Model:                            OLS   Adj. R-squared:                  0.550
Method:                 Least Squares   F-statistic:                     59.54
Date:                Tue, 08 Oct 2024   Prob (F-statistic):           5.88e-92
Time:                        14:51:58   Log-Likelihood:                 1465.4
No. Observations:                 576   AIC:                            -2905.
Df Residuals:                     563   BIC:                            -2848.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

## Residual Analysis

In [388]:
def residual_analysis(model, X, y):

    # Add the constant to X
    X_pred = sm.add_constant(X)

    # Create y_pred
    y_pred = model.predict(X_pred)

    # Create the dataframe of prediction
    df_pred = pd.DataFrame(data={"Observed": y.values.ravel(),
                                 "Prediction": y_pred.ravel()})

    # Create residual column
    df_pred["Residual"] = df_pred.Observed - df_pred.Prediction

    # Create a 2x2 subplot figure with reduced vertical spacing
    fig = make_subplots(rows=2, cols=2,
                        subplot_titles=("Observed vs. Predicted", 
                                        "Residuals vs. Predicted Values",
                                        "Histogram of Residuals", 
                                        "Q-Q Plot of Residuals"),
                        vertical_spacing=0.1)  # Decrease vertical space between rows

    # 1. Observed vs. Predicted Plot
    fig.add_trace(go.Scatter(x=df_pred['Observed'], y=df_pred['Prediction'],
                             mode='markers', name='Predicted vs Observed',
                             marker=dict(size=10, color='blue')), row=1, col=1)
    fig.add_trace(go.Scatter(x=df_pred['Observed'], y=df_pred['Observed'],
                             mode='lines', name='Perfect Prediction',
                             line=dict(color='red', dash='dash')), row=1, col=1)

    # 2. Residuals Plot
    fig.add_trace(go.Scatter(x=df_pred['Prediction'], y=df_pred['Residual'],
                             mode='markers', name='Residuals',
                             marker=dict(size=10, color='green')), row=1, col=2)
    fig.add_trace(go.Scatter(x=[df_pred['Prediction'].min(), df_pred['Prediction'].max()],
                             y=[0, 0], mode='lines', name='Zero Residual Line',
                             line=dict(color='red', dash='dash')), row=1, col=2)

    # 3. Histogram of Residuals (bins increased to 12)
    fig.add_trace(go.Histogram(x=df_pred['Residual'], name='Residuals',
                               marker=dict(color='orange'), nbinsx=12), row=2, col=1)

    # 4. Q-Q Plot of Residuals
    residuals = df_pred['Residual']
    residuals_sorted = np.sort(residuals)
    quantiles = np.linspace(0, 1, len(residuals_sorted))
    theoretical_quantiles = stats.norm.ppf(quantiles)

    fig.add_trace(go.Scatter(x=theoretical_quantiles, y=residuals_sorted,
                             mode='markers', name='Q-Q Plot',
                             marker=dict(size=10, color='purple')), row=2, col=2)
    fig.add_trace(go.Scatter(x=[theoretical_quantiles.min(), theoretical_quantiles.max()],
                             y=[theoretical_quantiles.min(), theoretical_quantiles.max()],
                             mode='lines', name='45 Degree Line',
                             line=dict(color='red', dash='dash')), row=2, col=2)

    # Update layout with new figure size and no legend
    fig.update_layout(title='Linear Regression Analysis',
                      height=1000, width=1200,  # Adjusting the figure size
                      showlegend=False)  # Disabling legend

    # Add x and y axis titles for each subplot
    fig.update_xaxes(title_text='Observed Values', row=1, col=1)
    fig.update_yaxes(title_text='Predicted Values', row=1, col=1)

    fig.update_xaxes(title_text='Predicted Values', row=1, col=2)
    fig.update_yaxes(title_text='Residuals', row=1, col=2)

    fig.update_xaxes(title_text='Residuals', row=2, col=1)
    fig.update_yaxes(title_text='Frequency', row=2, col=1)

    fig.update_xaxes(title_text='Theoretical Quantiles', row=2, col=2)
    fig.update_yaxes(title_text='Sample Quantiles', row=2, col=2)

    # Show the figure
    fig.show()

In [389]:
residual_analysis(lin1, X1, y1)

In [390]:
residual_analysis(lin2, X2, y2)

In [391]:
residual_analysis(lin3, X3, y3)

## Beta Regression - StatsModels

In [421]:
def create_beta_regression(X, y, normalize = False, columns_remove = None):

    y_copy = y.copy(deep = True)
    X_copy = X.copy(deep = True)

    if normalize == True:
        scaler = MinMaxScaler()
        y_copy = pd.DataFrame(scaler.fit_transform(y_copy), columns = ["score_test"])

    if columns_remove != None:
        X_copy = X_copy.drop(columns = columns_remove)

    #rename the columns
    X_copy = X_copy.rename(columns = {"time_interval_5.0": "time_interval_5",
                            "time_interval_10.0": "time_interval_10",
                            "time_interval_20.0": "time_interval_20",
                            "list_lag_features_[3]": "list_lag_features_3",
                            "list_lag_features_[5]": "list_lag_features_5"})
    
    y_copy.reset_index(drop = True, inplace = True)
    X_copy.reset_index(drop = True, inplace = True)

    # Fit a beta regression using GLM with a Binomial family (logit link)
    model_beta = glm(formula='score_test ~ time_interval_5 + time_interval_10 + time_interval_20 + \
                list_lag_features_3 + list_lag_features_5 + \
                remove_correlation_True + apply_fa_True + \
                apply_rebalance_strategy_over_sampling +  apply_rebalance_strategy_under_sampling + \
                model_LGBM + model_RF + model_XGB', 
                data=pd.concat([X_copy, y_copy], axis = 1), 
                family=InverseGaussian(link=log())).fit()

    # Print the model summary
    print(model_beta.summary())
    print("AIC: ", model_beta.aic)
    print("BIC: ", model_beta.bic)

    return model_beta, X_copy, y_copy

Linkfunction = Log

In [422]:
model_beta1, X1_after, y1_after = create_beta_regression(X1, y1, normalize = True)

                 Generalized Linear Model Regression Results                  
Dep. Variable:             score_test   No. Observations:                  573
Model:                            GLM   Df Residuals:                      560
Model Family:         InverseGaussian   Df Model:                           12
Link Function:                    log   Scale:                         0.18790
Method:                          IRLS   Log-Likelihood:                    nan
Date:                Tue, 08 Oct 2024   Deviance:                          inf
Time:                        14:58:25   Pearson chi2:                     105.
No. Iterations:                     1   Pseudo R-squ. (CS):                nan
Covariance Type:            nonrobust                                         
                                              coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

In [407]:
model_beta2, X2_after, y2_after = create_beta_regression(X2, y2, normalize = False)

                 Generalized Linear Model Regression Results                  
Dep. Variable:             score_test   No. Observations:                  576
Model:                            GLM   Df Residuals:                      563
Model Family:                   Gamma   Df Model:                           12
Link Function:                    log   Scale:                      0.00077639
Method:                          IRLS   Log-Likelihood:                 1390.0
Date:                Tue, 08 Oct 2024   Deviance:                      0.44333
Time:                        14:57:12   Pearson chi2:                    0.437
No. Iterations:                     8   Pseudo R-squ. (CS):             0.6674
Covariance Type:            nonrobust                                         
                                              coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

In [408]:
model_beta3, X3_after, y3_after = create_beta_regression(X3, y3, normalize = False)

                 Generalized Linear Model Regression Results                  
Dep. Variable:             score_test   No. Observations:                  576
Model:                            GLM   Df Residuals:                      563
Model Family:                   Gamma   Df Model:                           12
Link Function:                    log   Scale:                      0.00041195
Method:                          IRLS   Log-Likelihood:                 1452.2
Date:                Tue, 08 Oct 2024   Deviance:                      0.23355
Time:                        14:57:14   Pearson chi2:                    0.232
No. Iterations:                     7   Pseudo R-squ. (CS):             0.6993
Covariance Type:            nonrobust                                         
                                              coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

In [396]:
residual_analysis(lin1, X1, y1)

In [420]:
residual_analysis(model_beta1, X1_after, y1_after)

# Data Prep - Decision Tree

In [55]:
#copy dataframe with columns
df_dt = df_exp[["Period", "time_interval", "list_lag_features", "remove_correlation", 
                 "apply_fa", "apply_rebalance_strategy", "model", "score_test"]].copy()

#fill missing values
df_dt.fillna("None", inplace = True)

#rename columns
df_dt.rename(columns = {"time_interval": "H4 Manual feature selection",
                        "list_lag_features": "H5 Feature engineering",
                        "remove_correlation": "H2 Correlation removal",
                        "apply_fa": "H3 Dimensionality reduction",
                        "apply_rebalance_strategy": "H7 Rebalancing Strategy",
                        "model": "H12 ML model",
                        "score_test": "Score Test"},
             inplace = True)

#replace values
df_dt.replace({'(0-100]': "first",
              '(100-200]': "second",
              '(200-300]': "third",
              "[3]": "D1",
              "[5]": "D2",
              "over_sampling": "Over-sampling",
              "under_sampling": "Under-sampling",
              False: "None",
              1: "B1",
              5: "B2",
              10: "B3",
              20: "B4",}, 
              inplace = True)

#transform the dataframe into string
df_dt = df_dt.astype(str)


#export values
df_dt.to_csv("data/features_regression_rpart.csv")

df_dt.head()

,Period,H4 Manual feature selection,H5 Feature engineering,H2 Correlation removal,H3 Dimensionality reduction,H7 Rebalancing Strategy,H12 ML model,Score Test
0,first,B1,D1,None,None,None,RF,0.790232644983009
1,first,B1,D1,None,None,None,XGB,0.7900862616229135
2,first,B1,D1,True,None,None,RF,0.7897367340079914
3,first,B1,D2,True,None,None,XGB,0.7897008850218454
4,first,B2,D1,True,None,None,RF,0.7896949101908212


In [52]:
for column in df_dt.columns:
    print(column, df_dt[column].unique())

Period ['first' 'second' 'third']
H4: Manual feature selection ['B1' 'B2' 'B3' 'B4']
H5: Feature engineering ['D1' 'D2' 'None']
H2: Correlation removal ['None' 'True']
H3: Dimensionality reduction ['None' 'True']
H7: Rebalancing Strategy ['None' 'Over-sampling' 'Under-sampling']
H12: ML Model ['RF' 'XGB' 'LGBM' 'DT']
Score Test ['0.790232644983009' '0.7900862616229135' '0.7897367340079914' ...
 '0.8784614810112401' '0.8772784644684268' '0.873391090033235']


# Others

In [399]:
# # Print coefficients
# print("Coefficients:\n", model.params)

# # Print p-values
# print("\nP-values:\n", model.pvalues)

# # Print R-squared
# print("\nR-squared:\n", model.rsquared)

# # Print confidence intervals
# print("\nConfidence Intervals:\n", model.conf_int())

# # Print Adjusted R-squared
# print("\nAdjusted R-squared:\n", model.rsquared_adj)

# # Print F-statistic
# print("\nF-statistic:\n", model.fvalue)

# # Print Prob (F-statistic)
# print("\nProb (F-statistic):\n", model.f_pvalue)

# # Print AIC (Akaike Information Criterion)
# print("\nAIC:\n", model.aic)

# # Print BIC (Bayesian Information Criterion)
# print("\nBIC:\n", model.bic)

In [400]:
# pd.concat([X1.rename(columns = {"time_interval_5.0": "time_interval_5",
#                                 "time_interval_10.0": "time_interval_10",
#                                 "time_interval_20.0": "time_interval_20",
#                                 "list_lag_features_[3]": "list_lag_features_3",
#                                 "list_lag_features_[5]": "list_lag_features_5"}).drop(columns = "const"), y1], axis = 1).to_csv("X1.csv")

# pd.concat([X2.rename(columns = {"time_interval_5.0": "time_interval_5",
#                                 "time_interval_10.0": "time_interval_10",
#                                 "time_interval_20.0": "time_interval_20",
#                                 "list_lag_features_[3]": "list_lag_features_3",
#                                 "list_lag_features_[5]": "list_lag_features_5"}).drop(columns = "const"), y2], axis = 1).to_csv("X2.csv")

# pd.concat([X3.rename(columns = {"time_interval_5.0": "time_interval_5",
#                                 "time_interval_10.0": "time_interval_10",
#                                 "time_interval_20.0": "time_interval_20",
#                                 "list_lag_features_[3]": "list_lag_features_3",
#                                 "list_lag_features_[5]": "list_lag_features_5"}).drop(columns = "const"), y3], axis = 1).to_csv("X3.csv")